# Sports Ticket Classification with Logistic Regression – Practice Skeleton

**Short name (GitHub):** `Bet_LogReg`  
**Pattern:** same L1-logistic lab as `Income_LogReg`, rebuilt for a sportsbook ticket file.  
**Data:** `data/sports_tickets.csv` (12,000 pre-game wagers, synthetic book).  
**Target:** `result` is `Won` or `Lost` — did the ticket cash?

This is an **education lab**, not a tipping service. The file is synthetic. Past tickets do not imply future edge. Gambling involves risk of loss.

Companion files: `Bet_LogReg_Solution.ipynb`, `Bet_LogReg_Reusable_Template.ipynb`, `Bet_LogReg_Cheatsheet.docx`, `Bet_LogReg_Project_Memo.docx`, `Bet_LogReg_Strategy_Guide.docx`, `Bet_LogReg_1Page_Summary_Report.docx`, `bet_logreg_flowchart.png`.

## Inline cheat-sheet (keep this cell visible)

See also **`Bet_LogReg_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Load | `pd.read_csv("data/sports_tickets.csv")` |
| Class mix | `df.result.value_counts(normalize=True)` |
| Dummies | `X = pd.get_dummies(df[cols], drop_first=True).astype(float)` |
| Binary target | `y = np.where(df.result == "Lost", 0, 1)` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=1)` |
| L1 LR (lesson) | `LogisticRegression(C=0.05, penalty="l1", solver="liblinear")` |
| Confusion | `[[TN, FP], [FN, TP]]` with positive = Won |
| Accuracy | `(TN+TP)/n` — compare to the Lost baseline (~0.61) |
| Coef table | `pd.DataFrame({"var": cols, "coef": model.coef_[0]})` |
| ROC / AUC | `roc_curve(y, p); roc_auc_score(y, p)` |
| CLV | cents better than the *closing* number; positive CLV is the sharp signature |

**Industry reading of the signs you should recover:** `clv_cents` +, `line_move` +, `public_pct` −, `injury_star` −.

## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load the ticket blotter

### Task 1.1

Read `data/sports_tickets.csv`. Print `head()`, `shape`, and `dtypes`.

Columns of interest:

* `clv_cents` — closing-line value in cents (positive = you beat the close)
* `public_pct` — share of tickets on this side
* `line_move` — close minus open, in points
* `hours_to_start`, `stake_usd`, `is_home`
* `sport`, `market` (spread / moneyline / total)
* extras for later: `injury_star`, `weather_flag`, `live_bet`, `is_favorite`

In [ ]:
# YOUR CODE HERE
df = None
print(df.head())
print(df.shape)

## 2. EDA and logistic-regression assumptions

### Task 2.1 — class imbalance

Print raw counts and the mix. A book that always fades the customer (predict Lost) is already right about 61% of the time. Write one sentence on why accuracy will flatter that policy.

In [ ]:
# YOUR CODE HERE
print(df.result.value_counts())
print(df.result.value_counts(normalize=True))

### Task 2.2 — dummy-encode the working feature set

```python
feature_cols = ["clv_cents", "public_pct", "line_move", "hours_to_start",
                "stake_usd", "is_home", "sport", "market"]
```

`get_dummies(..., drop_first=True)`, cast to `float`, print shape and columns. Dropped sport reference is MLB; dropped market reference is moneyline.

In [ ]:
feature_cols = [
    "clv_cents", "public_pct", "line_move", "hours_to_start",
    "stake_usd", "is_home", "sport", "market",
]
# YOUR CODE HERE
X = None
print(X.shape)
print(list(X.columns))

### Task 2.3 — correlation heatmap

`sns.heatmap(X.corr())`. Sport dummies are mutually exclusive so they anti-correlate. Nothing here should look like a 0.99 clone.

In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(10, 8))
plt.title("Feature correlation (dummy-encoded X)")
plt.tight_layout()
plt.show()

### Task 2.4 — do we need to scale? Then encode `y`

Print min / max / mean for `clv_cents`, `public_pct`, `line_move`, `hours_to_start`, `stake_usd`.

Liblinear L1 will converge unscaled. Stake is in dollars and CLV is in cents, so the L1 budget is *not* fair across features.

`y = 1` when the ticket **Won**.

In [ ]:
# YOUR CODE HERE
for c in ["clv_cents", "public_pct", "line_move", "hours_to_start", "stake_usd"]:
    print(c, X[c].min(), X[c].max(), round(X[c].mean(), 2))

y = None
print("cash rate", y.mean())

## 3. Fit the lesson model

### Task 3.1 — split + L1 logistic regression

`random_state=1`, `test_size=0.2`, `LogisticRegression(C=0.05, penalty="l1", solver="liblinear")`.

In [ ]:
# YOUR CODE HERE
x_train = x_test = y_train = y_test = None
log_reg = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
y_pred = None
print(x_train.shape, x_test.shape, y_train.mean(), y_test.mean())

### Task 3.2 — intercept and coefficients

In [ ]:
print("Model Parameters, Intercept:")
# YOUR CODE HERE
print("Model Parameters, Coeff:")
# YOUR CODE HERE

### Task 3.3 — confusion matrix and accuracy

Ballpark on this split: accuracy near **0.63**, but Won-class recall near **0.21** at t = 0.5. The model is a conservative hold filter, not a steam-chasing bot.

In [ ]:
print("Confusion Matrix on test set:")
# YOUR CODE HERE
print("Accuracy Score on test set:")
# YOUR CODE HERE

## 4. Coefficient table and bar plot

### Task 4.1

DataFrame `coef_df` with `var`, `coef`. Drop exact zeros. Sort ascending.

In [ ]:
# YOUR CODE HERE
coef_df = None
print(coef_df)

### Task 4.2 — bar plot

Title: `LR Coefficient Values (ticket won)`.

In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(10, 6))
plt.xticks(rotation=90)
plt.title("LR Coefficient Values (ticket won)")
plt.tight_layout()
plt.show()

## 5. ROC curve and AUC

Sports ticket models that look “good” on social media often have AUC 0.52–0.55 after juice. Anything near 0.63 on *held-out* tickets is a weak screen, not a money printer.

In [ ]:
# YOUR CODE HERE
y_pred_prob = None
roc_auc = None
print("ROC AUC score:", roc_auc)

## 6. Alternate code

### Task 6.1 — scaled L1 pipeline

`Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=0.05, penalty="l1", solver="liblinear"))])`.

Compare AUC. Read the scaled `clv_cents` coefficient (now “per 1 SD of CLV”).

In [ ]:
# YOUR CODE HERE
pipe = None

### Task 6.2 — L2 default vs L1

`LogisticRegression(max_iter=2000)`. How many exact zeros vs L1?

In [ ]:
# YOUR CODE HERE
log_l2 = None

### Task 6.3 — CLV-only card

Fit L1 using only `clv_cents` as X (reshape to 2-D). How much AUC do the other features add?

In [ ]:
# YOUR CODE HERE


### Task 6.4 — threshold as a *hold percentage*

`predict_at(p, t)`. Sweep t = 0.30, 0.40, 0.50, 0.60. In a book, a high t is “only post the sides we really like”; a low t is “almost never shade the customer off.”

In [ ]:
def predict_at(proba, t=0.5):
    # YOUR CODE HERE
    return None

for t in (0.30, 0.40, 0.50, 0.60):
    pass

## 7. More practice

### Task 7.1 — add injury, weather, live, favorite

Rebuild X with those four columns added. Which new coefficient is largest in magnitude? What happens to AUC?

In [ ]:
# YOUR CODE HERE


### Task 7.2 — `class_weight="balanced"`

Keep the lesson feature set. Report Won-recall vs the default model. Accuracy will drop — that is a more aggressive posting policy.

In [ ]:
# YOUR CODE HERE


### Task 7.3 — sport slice

On the lesson test fold, compute Won-recall separately for NFL vs NBA vs the rest. A gap is a *desk* diagnostic (different juice, different public), not proof the sport is “beatable.”

In [ ]:
# YOUR CODE HERE


### Task 7.4 — which metric when?

One sentence each:

* trading desk deciding which 15% of tickets to *limit*
* marketing pushing a “hot pick” push notification
* finance asking “how often is the model right” for a board pack

In [ ]:
limit_desk = """..."""
hot_pick = """..."""
board_pack = """..."""
print(limit_desk); print(hot_pick); print(board_pack)

## 8. Simulation (edit the boxed parameters)

Each replicate draws `N` tickets with replacement, optionally flips a `NOISE` fraction of *training* labels (mis-keyed results / voided games), fits lesson L1, and scores at threshold `T`.

In [ ]:
# --- editable parameters ---
C = 0.05
N = 6000
N_REPS = 12
NOISE = 0.00      # train-label flip rate (voided / mis-keyed tickets)
T = 0.50
SEED = 1
# ---------------------------

rng = np.random.default_rng(SEED)
rows = []
# YOUR CODE HERE — fill `rows` then plot acc / recall / auc
sim = pd.DataFrame(rows)
print(sim.describe().round(3) if len(sim) else "no rows yet")

## 9. Audience rewrite

Using Jočys (data literacy, subject knowledge) and McMurrey (expert / technician / executive / nonspecialist), rewrite **one finding**: *unscaled L1 logistic regression on CLV, public %, line move, hours, stake, home, sport and market reaches test accuracy ≈ 0.63 and AUC ≈ 0.63, but Won-recall at t = 0.5 is only ≈ 0.21.*

Four seats: **trader/quant**, **trading-ops analyst**, **desk head**, **recreational bettor**.

In [ ]:
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)

## 10. When this project is a good fit — and when it is not

Ten good-fit settings and five anti-applications. Think juice, CLV, leaked closing lines, problem-gambling duty of care, and the fact that a 0.63 AUC does not pay the juice.

In [ ]:
good_fit = []
not_a_fit = []
for row in good_fit: print(row)
print("--- not a fit ---")
for row in not_a_fit: print(row)

## 11. Done checklist

- [ ] 12,000-row blotter loaded
- [ ] ~61 / 39 imbalance named against the Lost baseline
- [ ] 12-column dummy matrix, heatmap, scale note
- [ ] L1 model, CM, accuracy, sparse coefs, ROC
- [ ] Alternates + threshold-as-hold-policy
- [ ] Injury card, balanced weights, sport slice
- [ ] Simulation knobs
- [ ] Four-audience rewrite + good-fit list
- [ ] No claim that this model beats a real book